In [1]:
# from xtquant import xtdatacenter as xtdc
from xtquant import xtdata
import pandas as pd
import re
xtdata.reconnect(port = 58610)

***** xtdata连接成功 2026-07-03 10:27:44*****
服务信息: {'tag': 'qmt_research', 'version': '1.0'}
服务地址: 127.0.0.1:58610
数据路径: E:\迅投极速交易终端睿智融科版\datadir
设置xtdata.enable_hello = False可隐藏此消息



In [41]:
def get_xt_trade_date_list(start_date: str, end_date: str, market: str = "SH") -> list[str]:
    
    def normalize_input_date(value: str) -> str:
        text = str(value).strip().replace("-", "")
        if not re.fullmatch(r"\d{8}", text):
            raise ValueError(f"日期格式错误: {value}, 应为 YYYYMMDD 或 YYYY-MM-DD")
        return text

    def normalize_xt_date(value) -> str:
        if isinstance(value, pd.Timestamp):
            return value.strftime("%Y%m%d")

        text = str(value).strip()

        if re.fullmatch(r"\d{8}", text):
            return text

        num = pd.to_numeric(value, errors="coerce")
        if pd.isna(num):
            return pd.Timestamp(value).strftime("%Y%m%d")

        num = int(num)
        if num > 10**12:
            return (pd.to_datetime(num, unit="ms") + pd.Timedelta(days=1)).strftime("%Y%m%d")
        if num > 10**9:
            return (pd.to_datetime(num, unit="s") + pd.Timedelta(days=1)).strftime("%Y%m%d")  

        raise ValueError(f"无法识别 xtdata 交易日格式: {value}")

    start = normalize_input_date(start_date)
    end = normalize_input_date(end_date)
    if start > end:
        raise ValueError(f"开始日不能晚于截止日: {start} > {end}")

    raw_dates = xtdata.get_trading_dates(
        market,
        start_time=start,
        end_time=end,
        count=-1,
    )

    trade_dates = sorted({
        normalize_xt_date(item)
        for item in raw_dates
    })

    return [d for d in trade_dates if start <= d <= end]

In [43]:
INDEX_CODE = '000300'

# 指定特定日期
tick_data_ls = []

# 指定区间
start_date = "20260618"
end_date = "20260624"
tick_data_ls = get_xt_trade_date_list(start_date, end_date)
tick_data_ls

['20260618', '20260622', '20260623', '20260624']

In [44]:
INDEX_META = {
    "000016": {"name": "上证50", "ths": "000016.SH", "futures": "IH00.CFE", "primary_etf": "510050.SH"},
    "000300": {"name": "沪深300", "ths": "000300.SH", "futures": "IF00.CFE", "primary_etf": "510300.SH"},
    "000905": {"name": "中证500", "ths": "000905.SH", "futures": "IC00.CFE", "primary_etf": "510500.SH"},
    "000852": {"name": "中证1000", "ths": "000852.SH", "futures": "IM00.CFE", "primary_etf": "512100.SH"},
    "000688": {"name": "科创50", "ths": "000688.SH", "futures": "IC00.CFE", "primary_etf": "588000.SH"},
}
index_meta = INDEX_META[INDEX_CODE]
xt_index_code = index_meta["ths"]

In [45]:
df_real_index_by_date = {}
_index_minute_raw_by_date = {}

for fitting_date in tick_data_ls:
    print(f"Downloading real index 1m tick: {xt_index_code} {fitting_date}")
    xtdata.download_history_data(
        stock_code=xt_index_code,
        period="1m",
        start_time=fitting_date,
        end_time=fitting_date,
    )

    raw = xtdata.get_market_data_ex(
        field_list=[],
        stock_list=[xt_index_code],
        period="1m",
        start_time=fitting_date,
        end_time=fitting_date,
        count=-1,
    )
    _index_minute_raw_by_date[fitting_date] = raw

    frame = raw.get(xt_index_code)
    if frame is None or frame.empty:
        raise RuntimeError(f"Empty XtQuant index minute data: {xt_index_code} {fitting_date}")
    frame = frame.copy()
    frame["time"] = pd.to_datetime(frame.index)
    frame.index = frame["time"]
    df_real_index_by_date[fitting_date] = frame


In [46]:
df_real_index_summary = pd.DataFrame([
    {
        "fitting_date": fitting_date,
        "rows": len(frame),
        "first_time": frame.index[0] if len(frame.index) else None,
        "last_time": frame.index[-1] if len(frame.index) else None,
        "columns": ",".join(map(str, frame.columns)),
    }
    for fitting_date, frame in df_real_index_by_date.items()
])
display(df_real_index_summary)
display(df_real_index_by_date[tick_data_ls[0]].head())


,fitting_date,rows,first_time,last_time,columns
0,20260618,241,2026-06-18 09:30:00,2026-06-18 15:00:00,"time,open,high,low,close,volume,amount,settele..."
1,20260622,241,2026-06-22 09:30:00,2026-06-22 15:00:00,"time,open,high,low,close,volume,amount,settele..."
2,20260623,241,2026-06-23 09:30:00,2026-06-23 15:00:00,"time,open,high,low,close,volume,amount,settele..."
3,20260624,241,2026-06-24 09:30:00,2026-06-24 15:00:00,"time,open,high,low,close,volume,amount,settele..."


,time,open,high,low,close,volume,amount,settelementPrice,openInterest,preClose,suspendFlag
time,,,,,,,,,,,
2026-06-18 09:30:00,2026-06-18 09:30:00,4916.044,4916.044,4916.044,4916.044,3168210,7.800045e+09,0.0,0,4931.386,0
2026-06-18 09:31:00,2026-06-18 09:31:00,4916.838,4928.697,4916.838,4928.697,7760799,2.048933e+10,0.0,0,4916.044,0
2026-06-18 09:32:00,2026-06-18 09:32:00,4928.732,4934.483,4928.732,4934.483,6437548,1.775202e+10,0.0,0,4928.697,0
2026-06-18 09:33:00,2026-06-18 09:33:00,4933.932,4937.254,4933.932,4935.390,5688563,1.624948e+10,0.0,0,4934.482,0
2026-06-18 09:34:00,2026-06-18 09:34:00,4935.257,4941.375,4932.872,4941.375,4734591,1.410284e+10,0.0,0,4935.390,0
